In [1]:
# Initial text
text = "Hello, world! This is a sample text for testing."
torture_text = "Hello, world! café پاکستان 日本語 🤖 ∇²ψ" # Cursed unicode string as a stress test

# Implementing a naive tokenizer

In [2]:
chars = sorted(set(text))

char_to_index = {char: index for index, char in enumerate(chars)}
index_to_char = {index: char for index, char in enumerate(chars)}

def naive_encode(text):
    # This function encodes the input text into a list of tokens.
    return [char_to_index[char] for char in text]

def naive_decode(tokens):
    # This function decodes the list of tokens back into the original text.
    return ''.join(index_to_char[token] for token in tokens)

encoded_text = naive_encode("Hello world! This is a test.")
print("Encoded:", encoded_text)
decoded_text = naive_decode(encoded_text)
print("Decoded:", decoded_text)


Encoded: [4, 8, 13, 13, 16, 0, 21, 16, 18, 13, 7, 1, 0, 5, 11, 12, 19, 0, 12, 19, 0, 6, 0, 20, 8, 19, 20, 3]
Decoded: Hello world! This is a test.


# Byte-level tokenizer


In [3]:
# Unicode string
# → UTF-8 bytes
# → integer tokens
# → UTF-8 bytes
# → original string

def encode_unicode_string(unicode_string):
    # Encode the unicode string into UTF-8 bytes
    utf8_bytes = unicode_string.encode('utf-8')
    
    # Convert each byte to an integer token
    tokens = list(utf8_bytes)
    
    return tokens

def decode_unicode_tokens(tokens):
    # Convert the list of integer tokens back to bytes
    utf8_bytes = bytes(tokens)
    
    # Decode the UTF-8 bytes back into a unicode string
    unicode_string = utf8_bytes.decode('utf-8')
    
    return unicode_string

encoded_torture_text = encode_unicode_string(torture_text)
print("Encoded torture text:", encoded_torture_text)
decoded_torture_text = decode_unicode_tokens(encoded_torture_text)
print("Decoded torture text:", decoded_torture_text)

Encoded torture text: [72, 101, 108, 108, 111, 44, 32, 119, 111, 114, 108, 100, 33, 32, 99, 97, 102, 195, 169, 32, 217, 190, 216, 167, 218, 169, 216, 179, 216, 170, 216, 167, 217, 134, 32, 230, 151, 165, 230, 156, 172, 232, 170, 158, 32, 240, 159, 164, 150, 32, 226, 136, 135, 194, 178, 207, 136]
Decoded torture text: Hello, world! café پاکستان 日本語 🤖 ∇²ψ


# BPE (byte-pair encoding)

In [ ]:
def merge_pair(tokens, pair, new_token):
    # Merge the specified pair in the list of tokens
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_token)
            i += 2  # Skip the next token since it's part of the merged pair
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

def get_pair_frequencies(tokens):
    # Count the frequency of each adjacent pair of tokens
    freq = {}
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        freq[pair] = freq.get(pair, 0) + 1
    return freq

# Starting and target vocabulary sizes
target_vocab_size = 512

def perform_merges(tokens, target_vocab_size):
    # Starting vocabulary size
    starting_vocab_size = 256
    vocab = {i: i for i in range(starting_vocab_size)}  # Initial vocabulary mapping
    merges = {}

    while (target_vocab_size > starting_vocab_size):
        freq = get_pair_frequencies(tokens)

        if not freq:
            break

        best_pair = max(freq, key=freq.get)
        merged_tokens = merge_pair(tokens, best_pair, starting_vocab_size)

        starting_vocab_size += 1
        merges[best_pair] = starting_vocab_size - 1
        tokens = merged_tokens

    return tokens, vocab, merges

# Initialize the vocabulary and merges

# Encode the torture text into tokens
tokens = encode_unicode_string(torture_text)


# Get the frequency of each pair

tokens, vocab, merges = perform_merges(tokens, target_vocab_size)
print("Final tokens:", tokens)


NameError: name 'vocab' is not defined

In [ ]:
# Tiny Shakespeare corpus
with open("../data/tiny_shakespeare.txt", encoding="utf-8") as f:
    corpus = f.read()

tokens = encode_unicode_string(corpus[:100_000])  # Limit to first 100,000 characters for demonstration

fin_tokens, vocab, merges = perform_merges(tokens, target_vocab_size)

print(f"Encoded corpus into {len(fin_tokens):,} tokens")


Encoded corpus into 47,586 tokens
